# Train results — R0

**Strategy R0: whole-plan regeneration (baseline)**

This notebook reproduces the `R0` column of the **Train results** tab in `Results.xlsx`, end to end. It:

1. Runs the repository's own analyzer (`scripts/analyze_repair_strategies.py`) on the `R0` training run (`experiments/subset_train-r0`).
2. Reads the analyzer's `overall_summary.csv` and `mechanism_summary.csv`.
3. Derives the **common-core** metrics (shared by every strategy) and the **R0-specific** metrics.
4. Writes the exact table to `Train_results_R0.xlsx`.

Everything is read-only analysis of an existing run: **no experiments, no model calls, no benchmark scoring.** The outputs below are produced by actually executing the cells.

## Parameters — edit these if your run lives elsewhere

`STRATEGY` is this notebook's strategy. `RUN_DIR` is the **training run folder** to analyze. If a new/rerun `subset_train` experiment is saved under a different folder name, change `RUN_DIR` below and re-run the whole notebook — the two summary CSVs and the output `.xlsx` will regenerate from that run.

In [1]:
# === EDIT THESE if your training run is in a different folder ===
STRATEGY = 'R0'
RUN_DIR  = 'experiments/subset_train-r0'   # the subset_train run this notebook analyzes
# ================================================================

## Step 0 — Setup and run the analyzer
We run from the repository root and point the analyzer at this strategy's training run. The analyzer writes its tables under a scratch output directory.

In [2]:
import subprocess, sys, os, csv
from pathlib import Path

# Resolve the repository root (the folder that contains 'scripts/' and 'experiments/').
ROOT = Path.cwd()
while not (ROOT / 'scripts' / 'analyze_repair_strategies.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'scripts' / 'analyze_repair_strategies.py').exists(), 'Run this from inside the repo.'
NB_DIR = Path.cwd()  # folder containing this notebook (before we chdir to repo root)
os.chdir(ROOT)
PY = sys.executable  # the kernel's interpreter (the project venv)
assert 'STRATEGY' in globals() and 'RUN_DIR' in globals(), 'Run the Parameters cell first.'

# Analyzer output is written here, NEXT TO THIS NOTEBOOK (not a temp dir),
# so you can open overall_summary.csv / mechanism_summary.csv yourself.
OUT_DIR = str(NB_DIR / 'analysis_output')

assert Path(RUN_DIR).is_dir(), f'Missing run folder: {RUN_DIR}'
print('Repo root :', ROOT)
print('Strategy  :', STRATEGY)
print('Run folder:', RUN_DIR)
print('Analyzer output dir:', OUT_DIR)

Repo root : /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-
Strategy  : R0
Run folder: experiments/subset_train-r0
Analyzer output dir: /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/analysis_output


In [3]:
# Run the repo's analyzer for this one strategy. This only reads the saved run.
cmd = [PY, 'scripts/analyze_repair_strategies.py',
       '--run', f'{STRATEGY}={RUN_DIR}',
       '--output-dir', OUT_DIR, '--no-figures']
print('$', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout.strip()[-1500:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-2000:])
assert res.returncode == 0, 'analyzer failed'
print('\nAnalyzer exit code:', res.returncode)

$ /Users/enruifung/.pyenv/versions/3.12.8/bin/python3 scripts/analyze_repair_strategies.py --run R0=experiments/subset_train-r0 --output-dir /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/analysis_output --no-figures


R0: source=experiments/subset_train-r0, declared/evidenced queries=60, attempt queries=59, proposals=331, matrix cells=360

Analyzer exit code: 0


### Step 0b — Keep only the two summary CSVs
The analyzer emits many files we don't use (and an empty `figures/` folder). This removes everything except `overall_summary.csv` and `mechanism_summary.csv`, so re-running never leaves clutter behind.

In [4]:
# Keep the analyzer output tidy: the analyzer writes ~18 files (+ an empty
# figures/ dir) per strategy, but this notebook only needs the two summary
# CSVs. Delete everything else so the folder holds just what's useful.
import shutil
strat_out = Path(OUT_DIR) / STRATEGY
KEEP = {'overall_summary.csv', 'mechanism_summary.csv'}
removed = 0
for item in strat_out.iterdir():
    if item.name in KEEP:
        continue
    if item.is_dir():
        shutil.rmtree(item)
    else:
        item.unlink()
    removed += 1
print(f'Cleaned {strat_out}: removed {removed} extra item(s); kept {sorted(KEEP)}')
print('Files now:', sorted(p.name for p in strat_out.iterdir()))

Cleaned /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/analysis_output/R0: removed 16 extra item(s); kept ['mechanism_summary.csv', 'overall_summary.csv']
Files now: ['mechanism_summary.csv', 'overall_summary.csv']


## Step 1 — Load the analyzer's summary tables

> **Where do these CSVs come from?** They are **not** pre-existing files in the
> repo. **Step 0 above created them** by running the analyzer. They are written to
> `train_result_analysis/analysis_output/<STRATEGY>/` (printed by Step 0). If you
> look before running Step 0, the folder will not exist yet — that's expected.

`overall_summary.csv` holds counts/means per run; `mechanism_summary.csv` holds the strategy-specific mechanism metrics.

In [5]:
def load_summary(path):
    d = {}
    with open(path) as f:
        for row in csv.DictReader(f):
            d[row['metric']] = row
    return d

overall = load_summary(f'{OUT_DIR}/{STRATEGY}/overall_summary.csv')
mech = load_summary(f'{OUT_DIR}/{STRATEGY}/mechanism_summary.csv')
print('overall_summary metrics:', len(overall))
print('mechanism_summary metrics:', len(mech))
print('\nmechanism metrics available:')
for k, v in mech.items():
    print(f'  {k}: count={v["count"]} valid_n={v["valid_n"]} rate={v["rate"]}')

overall_summary metrics: 85
mechanism_summary metrics: 4

mechanism metrics available:
  plan_scope_proposals: count=331 valid_n=331 rate=1
  four_or_more_measured_changed_slots: count=53 valid_n=285 rate=0.185964912280702
  multiple_declared_personal_targets: count=36 valid_n=331 rate=0.108761329305136
  commonsense_proposals_regressing_personal: count=15 valid_n=144 rate=0.104166666666667


## Step 2 — Common-core metrics (shared by every strategy)
These come straight from `overall_summary.csv`. 'Confirmed unchanged attempts' is shown as `N (pct%)` of total attempts, matching the R1 template already in the spreadsheet.

In [6]:
def iv(metric):
    return int(float(overall[metric]['value']))

queries              = iv('declared_queries')
repair_attempts      = iv('total_attempts')
personal_attempts    = iv('personal_attempts')
commonsense_attempts = iv('commonsense_attempts')
unchanged            = iv('unchanged_confirmed')
unchanged_pct        = round(unchanged / repair_attempts * 100, 2)
personal_fixes       = iv('personal_fixed_count_sum')
personal_regressions = iv('personal_newly_violated_count_sum')

core = [
    ('Queries', queries),
    ('Repair attempts', repair_attempts),
    ('Personal repair attempts', personal_attempts),
    ('Commonsense repair attempts', commonsense_attempts),
    ('Confirmed unchanged attempts', f'{unchanged} ({unchanged_pct}%)'),
    ('Personal constraint fixes', personal_fixes),
    ('Personal regressions', personal_regressions),
]
for k, v in core:
    print(f'  {k:40} {v}')

  Queries                                  60
  Repair attempts                          331
  Personal repair attempts                 187
  Commonsense repair attempts              144
  Confirmed unchanged attempts             222 (67.07%)
  Personal constraint fixes                14
  Personal regressions                     2


## Step 3 — R0-specific metrics
These describe R0's own mechanism, read from `mechanism_summary.csv`. Each is formatted `count / valid_n (rate%)`.

In [7]:
SPECIFIC = [('Plan-scope proposals (always whole-plan)', 'plan_scope_proposals', 'count_rate'), ('Proposals changing >=4 slots', 'four_or_more_measured_changed_slots', 'count_rate'), ('Attempts with multiple declared personal targets', 'multiple_declared_personal_targets', 'count_rate'), ('Commonsense proposals that regressed a personal constraint', 'commonsense_proposals_regressing_personal', 'count_rate')]

def fmt_count_rate(row):
    c = int(float(row['count']))
    n = int(float(row['valid_n']))
    r = float(row['rate']) * 100
    return f'{c} / {n} ({r:.1f}%)'

specific = []
for label, metric, kind in SPECIFIC:
    if metric not in mech:
        specific.append((label, 'n/a (metric not produced for this run)'))
        continue
    row = mech[metric]
    specific.append((label, fmt_count_rate(row) if kind == 'count_rate' else row['rate']))
for k, v in specific:
    print(f'  {k:60} {v}')

  Plan-scope proposals (always whole-plan)                     331 / 331 (100.0%)
  Proposals changing >=4 slots                                 53 / 285 (18.6%)
  Attempts with multiple declared personal targets             36 / 331 (10.9%)
  Commonsense proposals that regressed a personal constraint   15 / 144 (10.4%)


In [8]:
# --- Load raw per-attempt and per-query data for analyses A-E ---
# (attempts.jsonl = one row per repair attempt; the per-query checkpoint JSON
#  carries initial_verification / final_verification and the repair_history.)
import json, glob
from collections import Counter

attempts = [json.loads(l) for l in open(Path(RUN_DIR, 'attempts.jsonl'), encoding='utf-8') if l.strip()]
ckpt_paths = sorted(glob.glob(str(Path(RUN_DIR, 'queries', '*', f'{STRATEGY}-*.json'))))
checkpoints = {}
for p in ckpt_paths:
    d = json.load(open(p, encoding='utf-8'))
    checkpoints[d['query_uid']] = d

def pc_pass_count(verif):
    # number of personal constraints that PASS (True) in a verification block
    pcs = (verif or {}).get('personal_constraints') or []
    return sum(1 for x in pcs if x is True)

print('attempts:', len(attempts), '| queries with checkpoints:', len(checkpoints))

attempts: 331 | queries with checkpoints: 60


### Analysis A — Initial vs final personal-constraint satisfaction
For each query, compare personal constraints passing in the **initial** plan vs the **final** plan (`Delta PC = PC_final - PC_initial`), and count how many queries improved, stayed the same, or worsened. This shows whether the final itinerary is actually better than the initial one (not just whether individual attempts fixed things).

In [9]:
improved = same = worsened = 0
delta_total = 0
for uid, d in checkpoints.items():
    pc_i = pc_pass_count(d.get('initial_verification'))
    pc_f = pc_pass_count(d.get('final_verification'))
    delta = pc_f - pc_i
    delta_total += delta
    if delta > 0: improved += 1
    elif delta < 0: worsened += 1
    else: same += 1
n_q = len(checkpoints)
analysisA = [
    ('[A] Queries where final PC improved vs initial', f'{improved} / {n_q}'),
    ('[A] Queries unchanged (final PC == initial)', f'{same} / {n_q}'),
    ('[A] Queries where final PC worsened vs initial', f'{worsened} / {n_q}'),
    ('[A] Net personal constraints gained (sum of deltas)', delta_total),
]
specific += analysisA
for k, v in analysisA: print(f'  {k:55} {v}')

  [A] Queries where final PC improved vs initial          5 / 60
  [A] Queries unchanged (final PC == initial)             47 / 60
  [A] Queries where final PC worsened vs initial          8 / 60
  [A] Net personal constraints gained (sum of deltas)     -13


### Analysis B — What happens when R0 changes the plan?
Classify every attempt into **mutually exclusive** outcomes based on the measured slot change (`changed_slot_count`): `>0` = confirmed change, `0` = confirmed no change, `None` = change not measurable for that attempt. Among confirmed changes we use a clear priority rule: an attempt that introduces any regression (`newly_violated`) is a regression even if it also fixed something; else if it fixed at least one (`fixed`) it's a fix; else it's neutral.

In [10]:
conf_unchanged = change_unmeasured = fixed_only = regressed = neutral = 0
for a in attempts:
    csc = a.get('changed_slot_count')
    if csc is None:
        change_unmeasured += 1
    elif csc == 0:
        conf_unchanged += 1
    else:  # csc > 0: a confirmed change
        if a.get('newly_violated'):
            regressed += 1
        elif a.get('fixed'):
            fixed_only += 1
        else:
            neutral += 1
N = len(attempts)
def pr(n): return f'{n} ({n / N * 100:.1f}%)'
analysisB = [
    ('[B] Attempts: confirmed no change (0 slots)', pr(conf_unchanged)),
    ('[B] Attempts: change not measurable', pr(change_unmeasured)),
    ('[B] Attempts: changed & fixed >=1 (no regression)', pr(fixed_only)),
    ('[B] Attempts: changed & introduced a regression', pr(regressed)),
    ('[B] Attempts: changed but neither fixed nor regressed', pr(neutral)),
]
specific += analysisB
assert conf_unchanged + change_unmeasured + fixed_only + regressed + neutral == N
for k, v in analysisB: print(f'  {k:55} {v}')

  [B] Attempts: confirmed no change (0 slots)             198 (59.8%)
  [B] Attempts: change not measurable                     46 (13.9%)
  [B] Attempts: changed & fixed >=1 (no regression)       11 (3.3%)
  [B] Attempts: changed & introduced a regression         16 (4.8%)
  [B] Attempts: changed but neither fixed nor regressed   60 (18.1%)


### Analysis C — Personal vs commonsense repair phase
Split attempts by `repair_phase` and, within each phase, report the share that fix at least one constraint, introduce a new violation, or make no confirmed change. This shows whether one phase is more productive than the other.

In [11]:
def phase_stats(phase):
    rows = [a for a in attempts if a.get('repair_phase') == phase]
    n = len(rows)
    if n == 0:
        return n, 'n/a', 'n/a', 'n/a'
    fix = sum(1 for a in rows if a.get('fixed'))
    viol = sum(1 for a in rows if a.get('newly_violated'))
    nochg = sum(1 for a in rows if not (isinstance(a.get('changed_slot_count'), (int, float)) and a['changed_slot_count'] > 0))
    f = lambda x: f'{x}/{n} ({x / n * 100:.1f}%)'
    return n, f(fix), f(viol), f(nochg)

analysisC = []
for phase in ('personal', 'commonsense'):
    n, fix, viol, nochg = phase_stats(phase)
    analysisC += [
        (f'[C] {phase.capitalize()} attempts (count)', n),
        (f'[C] {phase.capitalize()}: fixed >=1 targeted constraint', fix),
        (f'[C] {phase.capitalize()}: introduced a new violation', viol),
        (f'[C] {phase.capitalize()}: no confirmed change', nochg),
    ]
specific += analysisC
for k, v in analysisC: print(f'  {k:55} {v}')

  [C] Personal attempts (count)                           187
  [C] Personal: fixed >=1 targeted constraint             14/187 (7.5%)
  [C] Personal: introduced a new violation                2/187 (1.1%)
  [C] Personal: no confirmed change                       169/187 (90.4%)
  [C] Commonsense attempts (count)                        144
  [C] Commonsense: fixed >=1 targeted constraint          0/144 (0.0%)
  [C] Commonsense: introduced a new violation             15/144 (10.4%)
  [C] Commonsense: no confirmed change                    75/144 (52.1%)


### Analysis D — How expensive is R0?
Average tokens, model calls, and repair time **per query**, over queries with complete token data (a query is excluded from the token average if any of its attempts has no recorded token count, e.g. a failed model call). R0 regenerates the whole itinerary, so this shows the computational cost of that flexibility.

In [12]:
from collections import defaultdict
per_q_tokens = defaultdict(int); per_q_calls = defaultdict(int)
per_q_time = defaultdict(float); per_q_complete = defaultdict(lambda: True)
for a in attempts:
    uid = a['query_uid']
    tok = a.get('total_tokens_delta')
    if tok is None:
        per_q_complete[uid] = False
    else:
        per_q_tokens[uid] += tok
    per_q_calls[uid] += a.get('model_calls') or 0
    per_q_time[uid] += a.get('repair_time_seconds') or 0.0

all_q = list({a['query_uid'] for a in attempts})
complete_q = [u for u in all_q if per_q_complete[u]]
def avg(d, keys): return (sum(d[k] for k in keys) / len(keys)) if keys else 0
analysisD = [
    ('[D] Queries with complete token data', f'{len(complete_q)} / {len(all_q)}'),
    ('[D] Avg tokens per query (complete-data queries)', f'{avg(per_q_tokens, complete_q):,.0f}'),
    ('[D] Avg model calls per query', f'{avg(per_q_calls, all_q):.1f}'),
    ('[D] Avg repair time per query (s)', f'{avg(per_q_time, all_q):.1f}'),
]
specific += analysisD
for k, v in analysisD: print(f'  {k:55} {v}')

  [D] Queries with complete token data                    55 / 59
  [D] Avg tokens per query (complete-data queries)        96,849
  [D] Avg model calls per query                           2.6
  [D] Avg repair time per query (s)                       96.0


### Analysis E — One real R0 trajectory (fix + new violation)
Find an actual attempt where R0 fixed at least one personal constraint **and** introduced a new one, and show the before/after of the affected constraints with readable family labels. This connects the statistics to the research motivation (cascading violations). It is added to the table as a single descriptive row; the full before/after is printed below.

In [13]:
from chinatravel.evaluation.constraint_families import classify_constraint

def family_of(descriptors, idx):
    # descriptors use 1-based constraint_index; return a readable family or the DSL
    for d in descriptors or []:
        if d.get('constraint_index') == idx:
            try:
                fam = classify_constraint(d).get('family')
            except Exception:
                fam = None
            return fam or d.get('hard_logic_py', f'constraint {idx}')[:60]
    return f'constraint {idx}'

traj = next((a for a in attempts if a.get('fixed') and a.get('newly_violated')), None)
if traj is None:
    analysisE = [('[E] Example fix-and-regress trajectory', 'none found in this run')]
    print('No attempt both fixed and newly-violated a constraint in this run.')
else:
    descs = traj.get('constraint_descriptors') or []
    fixed_fams = [family_of(descs, i) for i in traj['fixed']]
    viol_fams = [family_of(descs, i) for i in traj['newly_violated']]
    uid = traj['query_uid']
    summary_cell = (f"query {uid}, step {traj['step']} ({traj['repair_phase']}): "
                    f"fixed {', '.join(fixed_fams)}; newly violated {', '.join(viol_fams)}")
    analysisE = [('[E] Example fix-and-regress trajectory', summary_cell)]
    print(f'Query {uid}  (attempt step {traj["step"]}, phase {traj["repair_phase"]})')
    print('  Before repair (personal constraints):', traj['before_personal_constraints'])
    print('  After  repair (personal constraints):', traj['after_personal_constraints'])
    for i in traj['fixed']:
        print(f'    constraint {i} [{family_of(descs, i)}]: Fail -> Pass  (fixed)')
    for i in traj['newly_violated']:
        print(f'    constraint {i} [{family_of(descs, i)}]: Pass -> Fail  (new violation)')
specific += analysisE

Query e20241028160908981620  (attempt step 1, phase personal)
  Before repair (personal constraints): [True, True, False, True, True, True, True, True]
  After  repair (personal constraints): [False, True, True, True, True, True, True, True]
    constraint 3 [monetary_budget]: Fail -> Pass  (fixed)
    constraint 1 [trip_specification]: Pass -> Fail  (new violation)


## Step 4 — Assemble the full table

In [14]:
rows = [('Metric', f'{STRATEGY} training result')]
rows += [(k, v) for k, v in core]
rows += [('', '')]  # spacer: everything ABOVE is common to every strategy;
                    # everything BELOW is R0-specific (mechanism + analyses A-E)
rows += [(k, v) for k, v in specific]

for k, v in rows:
    print(f'{k:60} | {v}')

Metric                                                       | R0 training result
Queries                                                      | 60
Repair attempts                                              | 331
Personal repair attempts                                     | 187
Commonsense repair attempts                                  | 144
Confirmed unchanged attempts                                 | 222 (67.07%)
Personal constraint fixes                                    | 14
Personal regressions                                         | 2
                                                             | 
Plan-scope proposals (always whole-plan)                     | 331 / 331 (100.0%)
Proposals changing >=4 slots                                 | 53 / 285 (18.6%)
Attempts with multiple declared personal targets             | 36 / 331 (10.9%)
Commonsense proposals that regressed a personal constraint   | 15 / 144 (10.4%)
[A] Queries where final PC improved vs initial            

## Step 5 — Write the exact table to `Train_results_R0.xlsx`

In [15]:
import openpyxl
from openpyxl.styles import Font

wb = openpyxl.Workbook()
ws = wb.active
ws.title = f'{STRATEGY} train results'

for r, (k, v) in enumerate(rows, start=1):
    ws.cell(r, 1, k)
    ws.cell(r, 2, v)

# bold the header row
ws.cell(1, 1).font = Font(bold=True)
ws.cell(1, 2).font = Font(bold=True)
ws.column_dimensions['A'].width = 52
ws.column_dimensions['B'].width = 40

out_xlsx = NB_DIR / f'Train_results_{STRATEGY}.xlsx'
wb.save(out_xlsx)
print('Wrote', out_xlsx)
print('Rows written:', len(rows))

Wrote /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/Train_results_R0.xlsx
Rows written: 35


## Step 6 — Read the file back to confirm it saved correctly

In [16]:
chk = openpyxl.load_workbook(out_xlsx, data_only=True)
cs = chk.active
print('Sheet:', cs.title, '| dims:', cs.dimensions)
print()
for row in cs.iter_rows(values_only=True):
    if any(c not in (None, '') for c in row):
        print(f'{str(row[0]):52} | {row[1]}')

Sheet: R0 train results | dims: A1:B35

Metric                                               | R0 training result
Queries                                              | 60
Repair attempts                                      | 331
Personal repair attempts                             | 187
Commonsense repair attempts                          | 144
Confirmed unchanged attempts                         | 222 (67.07%)
Personal constraint fixes                            | 14
Personal regressions                                 | 2
Plan-scope proposals (always whole-plan)             | 331 / 331 (100.0%)
Proposals changing >=4 slots                         | 53 / 285 (18.6%)
Attempts with multiple declared personal targets     | 36 / 331 (10.9%)
Commonsense proposals that regressed a personal constraint | 15 / 144 (10.4%)
[A] Queries where final PC improved vs initial       | 5 / 60
[A] Queries unchanged (final PC == initial)          | 47 / 60
[A] Queries where final PC worsened vs initial 